**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Quantum Computing for Signal Processors

A hype-resistant introduction with a home-field advantage: qubits are [unit vectors](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb), gates are unitary matrices, and the QFT — the algorithm behind quantum's most famous speedups — is *literally the FFT's matrix* ([verified](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) against `np.fft`). Everything simulated exactly in NumPy.

## 1. Pre-requisites

[Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) (unitary matrices, tensor structure helps), [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S7 (the DFT).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# a state of n qubits = a unit vector in C^(2^n); gates = unitaries; measurement = |amplitude|²
def kron_all(mats):
    out = np.array([[1.0+0j]])
    for m in mats: out = np.kron(out, m)
    return out
I2 = np.eye(2); H = np.array([[1, 1], [1, -1]])/np.sqrt(2)
X = np.array([[0, 1], [1, 0]]);
def phase(theta): return np.diag([1, np.exp(1j*theta)])

---
### 🕐 Session 1 of 3 — *Qubits Are Vectors, Gates Are Unitaries* (~35 min)
**Goal:** the whole formalism in linear-algebra terms; entanglement as non-factorizability.
**Builds on:** [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb). &nbsp; **Feeds into:** Session 2 (the QFT).

---

## 2. No Mysticism Required

💡 **Intuition.** One qubit: a unit vector in $\mathbb{C}^2$. $n$ qubits: a unit vector in $\mathbb{C}^{2^n}$ — the exponential size of that space is the entire hardware story. Gates are [unitary matrices](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) (reversible, norm-preserving — Parseval's cousins); measurement samples index $k$ with probability $|\psi_k|^2$ and is the *only* nonlinear thing in the theory. **Entanglement** is simply a joint state that doesn't factor as a tensor product — correlation with no classical joint distribution behind it.

In [2]:
# build the Bell state with H then CNOT; verify it cannot factor
CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
psi = np.zeros(4, complex); psi[0] = 1                      # |00⟩
psi = CNOT @ kron_all([H, I2]) @ psi
print("Bell state amplitudes:", psi.round(3), " → P(00)=P(11)=1/2, P(01)=P(10)=0")
# factorization test: a product state has rank-1 'amplitude matrix'
Mamp = psi.reshape(2, 2)
print(f"singular values of the amplitude matrix: {np.linalg.svd(Mamp, compute_uv=False).round(3)}")
print("→ TWO nonzero singular values: not rank-1 ⇒ genuinely entangled (the SVD detects it!)")

Bell state amplitudes: [0.707+0.j 0.   +0.j 0.   +0.j 0.707+0.j]  → P(00)=P(11)=1/2, P(01)=P(10)=0
singular values of the amplitude matrix: [0.707 0.707]
→ TWO nonzero singular values: not rank-1 ⇒ genuinely entangled (the SVD detects it!)


---
### 🕐 Session 2 of 3 — *The QFT Is the FFT* (~40 min)
**Goal:** build the quantum Fourier transform from gates; verify it equals the DFT matrix exactly.
**Builds on:** Session 1; [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S7. &nbsp; **Feeds into:** Session 3 (what quantum actually speeds up).

---

## 3. Home Turf

💡 **Intuition.** The QFT on $n$ qubits applies the $2^n \times 2^n$ **DFT matrix** to the amplitude vector — built from $O(n^2)$ two-qubit gates, the same divide-and-conquer as the [radix-2 FFT](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) (the gate cascade IS the butterfly diagram). The catch every headline omits: the result lives in *amplitudes you cannot read out directly* — measuring gives one sample, not the spectrum. QFT speedups exist only where a global *property* of the spectrum (like a period) suffices — which is exactly what Shor's algorithm extracts.

In [3]:
def qft_circuit(n_q):
    """QFT as a product of 1- and 2-qubit gates (returns the full unitary)."""
    N = 2**n_q
    U = np.eye(N, dtype=complex)
    for j in range(n_q):
        # H on qubit j
        U = kron_all([I2]*j + [H] + [I2]*(n_q-j-1)) @ U
        # controlled phases from qubits j+1..n
        for k in range(j+1, n_q):
            CP = np.eye(N, dtype=complex)
            for idx in range(N):
                bits = [(idx >> (n_q-1-b)) & 1 for b in range(n_q)]
                if bits[j] and bits[k]:
                    CP[idx, idx] = np.exp(2j*np.pi / 2**(k-j+1))
            U = CP @ U
    # bit-reversal permutation (same one as the FFT!)
    perm = [int(format(i, f"0{n_q}b")[::-1], 2) for i in range(N)]
    return U[perm]

n_q = 4
U_qft = qft_circuit(n_q)
F = np.array([[np.exp(2j*np.pi*i*k/2**n_q) for k in range(2**n_q)] for i in range(2**n_q)])/np.sqrt(2**n_q)
print(f"ORACLE: ‖QFT circuit − DFT matrix‖∞ = {np.abs(U_qft - F).max():.2e}")
assert np.abs(U_qft - F).max() < 1e-10
print(f"and against np.fft: ‖U_qft @ e₃ − ifft-convention column‖ = "
      f"{np.abs(U_qft[:, 3] - np.fft.ifft(np.eye(16)[3])*4).max():.2e}")

ORACLE: ‖QFT circuit − DFT matrix‖∞ = 3.78e-15
and against np.fft: ‖U_qft @ e₃ − ifft-convention column‖ = 1.49e-16


In [4]:
# period finding — the heart of Shor — on a simulated register
n_q = 6; N = 2**n_q
r_period = 8
psi = np.zeros(N, complex)
psi[::r_period] = 1; psi /= np.linalg.norm(psi)             # a periodic state (post-oracle)
out = qft_circuit(n_q) @ psi
probs = np.abs(out)**2
plt.figure(figsize=(7.5, 2.4))
plt.stem(probs, basefmt=" ", markerfmt=".")
plt.title(f"measure after QFT: peaks at multiples of N/r = {N//r_period} → the period, from ONE global property")
plt.xlabel("measured value"); plt.tight_layout(); plt.show()
peaks = np.where(probs > 0.01)[0]
print(f"measurement outcomes: {[int(p) for p in peaks]} — spacing {N//r_period} reveals r = {r_period}")

measurement outcomes: [0, 8, 16, 24, 32, 40, 48, 56] — spacing 8 reveals r = 8


/tmp/ipykernel_319673/2780989073.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("measured value"); plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *What Quantum Actually Speeds Up* (~30 min)
**Goal:** the honest scoreboard: where proofs exist, where hype lives, and what to watch.
**Builds on:** Session 2.

---

## 4. The Scoreboard

| Problem | Speedup | Status |
|---|---|---|
| Factoring / discrete log (Shor) | exponential | proven; needs ~millions of good qubits |
| Unstructured search (Grover) | quadratic only | proven; modest in practice |
| Simulating quantum systems | exponential | the original killer app — chemistry/materials |
| Generic ML / optimization | — | **no proven advantage**; data loading often eats the win |

💡 **Intuition.** The honest summary for an engineer: quantum computers are *interference machines* — they win when a problem's answer can be encoded so wrong paths cancel ([the QFT's](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) specialty) — and today's hardware fights decoherence with error-correction overheads of ~1000 physical per logical qubit. Track logical-qubit counts, not press releases. Your DSP training transfers verbatim: unitaries, interference, transforms — you already speak the language.

---
## Where next

- [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S7 — the butterfly you just rebuilt from gates.
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) — the entire formalism, secretly.